# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² dataset using the [`mlcroissant`](https://mlcommons.github.io/data/package/mlcroissant/) library. We work step-by-step from discovering available record sets to transforming and visualizing clinical data.

### Dataset Source
The dataset metadata and structure are provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant pandas matplotlib seaborn --quiet

## 1. Data Loading

We'll load the dataset's Croissant schema and extract the metadata as well as explore available record sets and fields.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define dataset schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access and print basic metadata
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"License: {metadata.license}")
print(f"Identifier: {metadata.identifier}")

## 2. Data Overview

Let's review the available record sets and their fields. We'll display each record set's `@id`, name, and description (where available), plus their field `@id`s.

In [ ]:
from collections import OrderedDict

# Query all available record sets
all_record_sets = list(dataset.record_sets())

print(f"Number of record sets: {len(all_record_sets)}\n")

record_set_ids = []

for record_set in all_record_sets:
    rid = record_set.id # Croissant `@id` for the record set
    record_set_ids.append(rid)
    print(f"Record set @id: {rid}")
    print(f"  Name: {getattr(record_set, 'name', None)}")
    print(f"  Description: {getattr(record_set, 'description', None)}")
    # List available fields and their @id
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.id} ({getattr(field, 'name', None)})")
    print()

# For demo/reproducibility, store the first record set id as default
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Defaulting to first record set: {main_record_set_id}")
else:
    main_record_set_id = None

## 3. Data Extraction

Load all available record sets into pandas DataFrames for easy manipulation.

We use each record set's `@id` and field `@id`s (referenced explicitly) to ensure robust data access.

In [ ]:
# Load all record sets by @id to DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    # Each record is an OrderedDict keyed by field @id
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"  -> {len(records)} records loaded.")

# Show available columns (field @id) and preview head for main record set
if main_record_set_id:
    print(f"\nFields (@id's) in main record set '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record set available to display.")

## 4. Exploratory Data Analysis (EDA)

Let's process numeric and categorical data. We'll perform steps such as:

- Filtering by a numeric field (e.g., age)
- Normalizing (z-score) that field
- Grouping by another attribute (e.g., sex/gender or cancer type)

All fields and record sets are referenced strictly by their `@id`.

In [ ]:
# ---- Customize field selection using exact @id's from the overview above ----
# For example, if age column @id is 'https://api.app.sen.science/frontiers/7862866/age', and sex is (...)/sex
# Adjust these variables as needed based on overview output.
# We'll set them after inspecting the real IDs from dataframes[main_record_set_id]

numeric_field_id = None
group_field_id = None
for col in dataframes[main_record_set_id].columns:
    lowcol = col.lower()
    if 'age' in lowcol and numeric_field_id is None:
        numeric_field_id = col
    if (('sex' in lowcol or 'gender' in lowcol) and group_field_id is None):
        group_field_id = col
    elif (('type' in lowcol or 'location' in lowcol) and group_field_id is None):
        group_field_id = col

print(f"Using numeric field @id: {numeric_field_id}")
print(f"Using group field @id: {group_field_id}")

# Convert numeric field to float, handling errors
df = dataframes[main_record_set_id].copy()

if numeric_field_id is not None:
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    # Set a threshold: filter age > minimum age (for demo: 40)
    threshold = 40
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    display(filtered_df[[numeric_field_id] + ([group_field_id] if group_field_id else [])].head())

    # Z-score normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} (z-score):")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id in filtered_df.columns:
        # Group and compute groupwise stats
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'std', 'count'])
        print(f"\nGrouped {numeric_field_id} statistics by {group_field_id}:")
        display(grouped)
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization

Let's plot the distribution of our numeric field (e.g., age) and show boxplots by group (e.g., sex or cancer type).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,4))
if numeric_field_id is not None and numeric_field_id in df.columns:
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True, color='coral')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

if numeric_field_id and group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=15)
    plt.show()

## 6. Conclusion

In this notebook, we have:

- Loaded the FAIR² dataset using the mlcroissant library and inspected its structure using only `@id` references.
- Accessed all available record sets and fields, loaded the main table into a DataFrame, and explored numeric and categorical variables directly by `@id`.
- Filtered, normalized, and grouped the data for initial insights.
- Visualized key distributions to support further clinical and biomarker research for second primary colorectal cancer in survivors.

For advanced analysis, repeat these steps with different record sets or fields, always referencing them by `@id` for reliability.